#### Agentic RAG System

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.agents import create_agent
from langchain.tools import tool

In [8]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)

In [10]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [6]:
vector_store = InMemoryVectorStore.from_documents(
    embedding=embeddings,
    documents=splitted_data
)

In [25]:
@tool
def retriver_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports.
    """
    
    print("function called", query)
    data = vector_store.similarity_search(query=query, k=4)
    context = ""
    for i in data:
        context += i.page_content + "\n"
        
    return context

In [26]:
llm = ChatOpenAI(model="gpt-5")

In [27]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [28]:
agent = create_agent(
    model=llm,
    tools=[retriver_tool],
    system_prompt=System_Prompt,
)

In [30]:
query = "What is the name of the Patient and, what is name of the doctors?"
res = agent.invoke({"messages":[{"role":"user", "content":query}]})

function called Doctor name
function called Patient name
function called Consultant
function called Physician
function called Attending physician


In [31]:
print(res["messages"][-1].content)

- Patient name: Ms. Nikita Choudhary
- Doctors:
  - Dr. Nitin Nahar (Referring physician)
  - Dr. Beena Chandrasekhar
  - Dr. Nagarjun Sai Jaine
  - Dr. Sunanda
  - Dr. Swapnil Gupta
  - Dr. Kiran Bhargava Pathak
